#### 1. Tokenizing to trigrams using `ntlk`

Installing libraries (if necessary)

In [ ]:
#%pip install nltk
import nltk
import os
import nltk
from nltk.collocations import TrigramCollocationFinder
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import os
from nltk import ngrams, FreqDist

def get_character_trigrams(filepath: str, min_freq: int = 1) -> list[tuple]:
    """
    Reads a preprocessed .txt file and extracts character trigrams using nltk.
    Returns a list of (trigram, frequency) tuples filtered by min_freq.
    """

    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read().strip()

    # Generate character trigrams
    trigrams = ngrams(text, 3)

    # Frequency distribution
    freq_dist = FreqDist(trigrams)

    return [(tri, freq) for tri, freq in freq_dist.items() if freq >= min_freq]


# Directory processing
txt_dir = "./preprocessed_landId/test"
output_dir = "./trigram_output"

# Create output folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

all_trigram_freqs = []

for filename in os.listdir(txt_dir):
    filepath = os.path.join(txt_dir, filename)

    file_trigrams = get_character_trigrams(filepath, min_freq=5)
    all_trigram_freqs.extend(file_trigrams)

    # Save trigrams to txt file
    output_filepath = os.path.join(output_dir, f"{filename}_trigrams.txt")

    with open(output_filepath, "w", encoding="utf-8") as f:
        for trigram, freq in file_trigrams:
            f.write(f"{''.join(trigram)}\t{freq}\n")

    print(f"{filename}: trobats {len(file_trigrams)} trigrames únics després del filtre")

deu_tst.txt: trobats 7796 trigrames únics després del filtre
eng_tst.txt: trobats 6590 trigrames únics després del filtre
fra_tst.txt: trobats 7238 trigrames únics després del filtre
ita_tst.txt: trobats 5625 trigrames únics després del filtre
nld_tst.txt: trobats 6817 trigrames únics després del filtre
spa_tst.txt: trobats 6287 trigrames únics després del filtre


In [11]:
from collections import Counter
import math

class TrigramLanguageModel:
    def __init__(self, lambda_smoothing=0.01):
        self.lambda_smoothing = lambda_smoothing
        self.language_counts = {}   # language -> Counter(trigram)
        self.language_totals = {}    # language -> total trigram count
        self.vocabulary = set()

    def train(self, language, trigram_list):
        """
        trigram_list → list of (trigram, frequency)
        """

        if language not in self.language_counts:
            self.language_counts[language] = Counter()

        for trigram, freq in trigram_list:
            self.language_counts[language][trigram] += freq
            self.vocabulary.add(trigram)

        self.language_totals[language] = sum(
            self.language_counts[language].values()
        )

    def trigram_probability(self, trigram, language):
        V = len(self.vocabulary)
        lambda_ = self.lambda_smoothing

        count = self.language_counts[language][trigram]
        total = self.language_totals[language]

        return (count + lambda_) / (total + lambda_ * V)

    def score_language(self, text_trigrams, language):
        """
        Compute log likelihood of text belonging to language
        """

        score = 0

        for trigram, freq in text_trigrams:
            prob = self.trigram_probability(trigram, language)
            score += freq * math.log(prob)

        return score

    def predict(self, filepath):
        """
        Predict language from a txt file path using the existing trigram extractor.
        """

        # Extract trigrams using your predefined function
        text_trigrams = get_character_trigrams(filepath, min_freq=1)

        scores = {}

        for language in self.language_counts:
            scores[language] = self.score_language(text_trigrams, language)

        return max(scores, key=scores.get)

In [12]:

def train_model_from_txt_folder(model, dataset_dir, min_freq=5):
    """
    Entrena el model quan la llengua està indicada
    per les 3 primeres lletres del nom del fitxer.
    """

    for filename in os.listdir(dataset_dir):

        if not filename.endswith(".txt"):
            continue

        # Les 3 primeres lletres = llengua
        language = filename[:3]

        filepath = os.path.join(dataset_dir, filename)

        print(f"Entrenant llengua: {language}")

        trigrams = get_character_trigrams(
            filepath,
            min_freq=min_freq
        )

        model.train(language, trigrams)

    print("Entrenament completat.")

In [18]:
model = TrigramLanguageModel(0.01)
train_model_from_txt_folder(model, "preprocessed_landId/train")

Entrenant llengua: deu
Entrenant llengua: eng
Entrenant llengua: fra
Entrenant llengua: ita
Entrenant llengua: nld
Entrenant llengua: spa
Entrenament completat.


In [19]:
model = TrigramLanguageModel(0.01)

prediction = model.predict("preprocessed_landId/test/eng_tst.txt")

print(prediction)

ValueError: max() iterable argument is empty